# 5. Modellierung der Daten

In [242]:
import re

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Memory, dump
from tqdm.auto import tqdm
from pathlib import Path

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sklearn_memory = Memory(
    location=project_root / "cache" / "sklearn",
    verbose=0,
)

model_output_dir = project_root / "models"
model_output_dir.mkdir(parents=True, exist_ok=True)


def safe_model_filename(name):
    """Erzeugt einen sicheren Dateinamen für gespeicherte Modelle."""
    safe_name = re.sub(r"[^0-9a-zA-Z_-]+", "_", name)
    return safe_name.strip("_").lower()


def fit_and_save_model(
    model,
    model_name,
    X,
    y,
    subdirectory,
):
    """Fitten auf allen Trainingsdaten und als Joblib-Datei speichern."""
    target_dir = model_output_dir / subdirectory
    target_dir.mkdir(parents=True, exist_ok=True)

    model_path = target_dir / f"{safe_model_filename(model_name)}.joblib"

    fitted_model = clone(model)
    fitted_model.fit(X, y)

    dump(fitted_model, model_path)

    return model_path

train_df = pd.read_csv(
    project_root / "data" / "processed" / "train_data.csv",
    sep=";",
    index_col=0,
    encoding="utf-8",
)

print("Train:", train_df.shape)

display(train_df.head())


d:\toydev\schwarz-test\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (45920, 28)


,name,geber,art,jahr,anschrift,politikbereich,zweck,betrag,empfaengerid,name_normalized,...,anschrift_standardised,zweck_standardised,empfaengerid_standardised,name_standardised_before_modelling,geber_standardised_before_modelling,art_standardised_before_modelling,anschrift_standardised_before_modelling,zweck_standardised_before_modelling,empfaengerid_standardised_before_modelling,betrag_standardised
id,,,,,,,,,,,,,,,,,,,,,
33655,Cashmere Radio e. V.,Senatsverwaltung für Kultur und Gesellschaftli...,Projektförderung,2024,"Frankfurter Allee 7, 10247 Berlin",Kultur,"signal2noise ? Art, Aesthetics And Social Prac...",120000,vr_035983,cashmere radio e. v.,...,"frankfurter allee 7, 10247 berlin-bezirk fried...","signal2noise ? art, aesthetics and social prac...",vr_035983,cashmere radio e. v.,senatsverwaltung für kultur und gesellschaftli...,projektförderung,"Frankfurter Allee 7, 10247 Berlin-Bezirk Fried...","signal2noise ? art, aesthetics and social prac...",vr_035983,11.695255
105910,Merantix Labs GmbH,"Senatsverwaltung für Wirtschaft, Energie und B...",Projektförderung,2022,"Max-Urich-Straße 3, 13355 Berlin",Wirtschaft,Errichtung einer Betriebsstätte,5481380,hrb_221397,merantix labs gmbh,...,"c/o ai campus, fachbereich bionik und evolutio...",errichtung einer betriebsstätte,hrb_221397,merantix labs gmbh,"senatsverwaltung für wirtschaft, energie und b...",projektförderung,"c/o AI Campus, Fachbereich Bionik und Evolutio...",errichtung einer betriebsstätte,hrb_221397,15.516868
67465,Georg-Kolbe-Stiftung,Senatsverwaltung für Kultur und Europa,Projektförderung,2020,"Sensburger Allee 25, 14055 Berlin",Kultur,Der absolute Tanz- Festival sculpture,70000,spr_100011,georg-kolbe-stiftung,...,"sensburger allee 25, 14055 berlin-bezirk charl...",der absolute tanz- festival sculpture,spr_100011,georg kolbe-stiftung,senatsverwaltung für kultur und europa,projektförderung,"Sensburger Allee 25, 14055 Berlin-Bezirk Charl...",der absolute tanz- festival sculpture,spr_100011,11.156265
159301,Verschiedene Gesellschaften bürgerlichen Rechts,"Senatsverwaltung für Inneres, Digitalisierung ...",Projektförderung,2023,'---',Sport,Kosten für die Beschäftigung von Übungsleitern,1380,NaN,verschiedene gesellschaften bürgerlichen rechts,...,deutschland,kosten für die beschäftigung von übungsleitern,NaN,verschiedene gesellschaften bürgerlichen rechts,"senatsverwaltung für inneres, digitalisierung ...",projektförderung,Deutschland,kosten für die beschäftigung von übungsleitern,NaN,7.230563
19495,Berliner Rugby-Club,Senatsverwaltung für Inneres und Sport,Projektförderung,2020,"Scharfestraße 12, 14169 Berlin",Sport,anteilige Finanzierung des Spielbetriebes der ...,9000,vr_002548,berliner rugby-club,...,"scharfestrasse 12, 14169 berlin-bezirk steglit...",anteilige finanzierung des spielbetriebes der ...,vr_002548,berliner rugby-club,senatsverwaltung für inneres und sport,projektförderung,"Scharfestraße 12, 14169 Berlin-Bezirk Steglitz...",anteilige finanzierung des spielbetriebes der ...,vr_002548,9.105091


## 5.1. Vorbereitung der Modellierung

Die Vorverarbeitung wird als scikit-learn `Pipeline` aufgebaut, weil dadurch alle lernenden Schritte innerhalb der Cross-Validation nur auf den jeweiligen Trainingsdaten angepasst werden. Das reduziert das Risiko von Data Leakage.

### 5.1.1. Zielvariable und Eingabemerkmale trennen

Zuerst werden Zielvariable und Eingabemerkmale getrennt. Als Zielvariable wird `politikbereich` verwendet. Für die Eingabemerkmale werden die bereits bereinigten und standardisierten Textspalten, die numerischen Spalte `betrag` sowie kategorische Spalten `art` und `jahr` genutzt. Die Empfänger-ID wird hier bewusst nicht als direktes Modellmerkmal verwendet, weil sie stark organisationsspezifisch ist und das Modell sonst eher bekannte Empfänger memorisieren könnte.


In [243]:
target_column = "politikbereich"

text_features = [
    "name_standardised",
    "geber_standardised",
    "anschrift_standardised",
    "zweck_standardised",
]

numeric_features = [
    "betrag",
]

categorical_features = [
    "art_standardised",
    "jahr",
]

feature_columns = text_features + numeric_features + categorical_features

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

missing_feature_values = (
    X_train
    .isna()
    .sum()
    .rename("missing_values")
    .reset_index()
    .rename(columns={"index": "feature"})
)

display(missing_feature_values)


,feature,missing_values
0,name_standardised,0
1,geber_standardised,0
2,anschrift_standardised,0
3,zweck_standardised,1
4,betrag,0
5,art_standardised,0
6,jahr,0


### 5.1.2. Textspalten vorbereiten

Die Textspalten können nicht direkt von klassischen scikit-learn-Modellen verarbeitet werden. Deshalb werden sie in TF-IDF-Merkmale umgewandelt. TF-IDF ist für diese Aufgabe geeignet, weil nicht nur das Vorkommen eines Wortes zählt, sondern auch seine Spezifität innerhalb des gesamten Korpus. Sehr seltene Terme werden mit `min_df=2` entfernt, da sie häufig Tippfehler oder Einzelfälle darstellen. Sehr häufige Terme werden mit `max_df=0.90` reduziert, weil sie meist wenig trennscharf sind. Mit `ngram_range=(1, 2)` werden zusätzlich Wortpaare berücksichtigt, z. B. fachlich relevante Ausdrücke wie „soziale stadt“.


TF: Term Frequency
IDF: Inverse Document Frequency

In [244]:
def flatten_column(values):
    """Convert a single-column 2D array into a 1D string array."""
    return np.asarray(values, dtype=object).ravel()


text_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="",
            ),
        ),
        (
            "flatten",
            FunctionTransformer(
                flatten_column,
                validate=False,
            ),
        ),
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                min_df=3,
                max_df=0.90,
                ngram_range=(1, 2),
                sublinear_tf=True,
                max_features=50_000,
                dtype=np.float32,
            ),
        ),
    ]
)

### 5.1.3. Numerische Spalten vorbereiten

Die numerischen Merkmale werden getrennt von den Textspalten verarbeitet. Fehlende Werte werden mit dem Median ersetzt, weil der Median robuster gegenüber extremen Förderbeträgen ist als der Mittelwert. Anschließend werden die numerischen Werte standardisiert. Das ist besonders für lineare Modelle wichtig, weil Merkmale mit großer Skala sonst einen unverhältnismäßig starken Einfluss bekommen können.


In [245]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)


### 5.1.3. Kategorische Spalten vorbereiten


Für die kategorialen Merkmale werden in diesem Notebook `art_standardised` und `jahr` verwendet. Die Spalte `art_standardised` beschreibt die Art der Förderung und besitzt keine natürliche numerische Reihenfolge. Auch `jahr` wird in diesem Modellierungsansatz nicht als kontinuierliche Zahl behandelt, sondern als diskrete Kategorie. Dadurch kann das Modell jahresspezifische Unterschiede berücksichtigen, ohne eine lineare zeitliche Entwicklung zwischen den Jahren vorauszusetzen.

Die Verarbeitung erfolgt mit einer scikit-learn `Pipeline`. Zunächst werden fehlende Werte mit `SimpleImputer(strategy="most_frequent")` durch die häufigste Kategorie ersetzt. Danach werden die Kategorien mit `OneHotEncoder` in binäre Merkmale umgewandelt. Für jede Ausprägung entsteht dadurch eine eigene Indikatorspalte. So kann das Modell kategoriale Informationen nutzen, ohne dass eine künstliche Ordnung zwischen den Kategorien entsteht.

Mit `handle_unknown="ignore"` bleibt die Pipeline außerdem stabil, falls in späteren Daten eine Kategorie auftritt, die im Training nicht vorhanden war. Diese unbekannte Kategorie führt dann nicht zu einem Fehler, sondern wird in der One-Hot-Repräsentation ignoriert.

In [246]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
        ),
    ]
)

### 5.1.4. Spaltenspezifische Vorverarbeitung zusammenführen

Da Text- und numerische Spalten unterschiedliche Verarbeitung benötigen, werden sie mit einem `ColumnTransformer` zusammengeführt. Jede Textspalte erhält eine eigene TF-IDF-Repräsentation, damit der Kontext der jeweiligen Spalte erhalten bleibt. Der Förderzweck, der Geber und der Empfängername werden dadurch nicht zu einem einzigen Textblock vermischt, sondern als getrennte Informationsquellen behandelt.


In [247]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            f"tfidf_{column}",
            text_transformer,
            [column],
        )
        for column in text_features
    ]
    + [
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        )
    ]
    + [
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        )
    ],
    remainder="drop",
)

print("Preprocessor definiert f\u00fcr", len(feature_columns), "Eingabespalten.")

Preprocessor definiert für 7 Eingabespalten.


## 5.2. Evaluationsrahmen vorbereiten

Wegen der Klassenunverteilung wird die spätere Modellbewertung nicht nur über Accuracy erfolgen. Accuracy zeigt zwar den Anteil aller korrekt vorhergesagten Fälle, kann bei unausgewogenen Klassen jedoch stark durch große Klassen dominiert werden. Deshalb werden zusätzliche Metriken verwendet, die die Leistung über alle Politikbereiche hinweg differenzierter bewerten.

Der Macro-F1-Score wird als Hauptmetrik verwendet, da er jede Klasse gleich stark gewichtet und dadurch auch Probleme bei kleinen Klassen sichtbar macht. Der Weighted-F1-Score ergänzt diese Sicht, indem er die tatsächliche Klassenhäufigkeit berücksichtigt. Zusätzlich wird Balanced Accuracy verwendet, weil sie die durchschnittliche Erkennungsrate über alle Klassen hinweg misst.

Metriken pro Klasse und die Konfusionsmatrix werden nicht direkt im Cross-Validation-Scoring berechnet, sondern später bei der detaillierten Modellbewertung ergänzt. Sie helfen dabei zu erkennen, welche Politikbereiche besonders häufig falsch klassifiziert werden und welche Klassen vom Modell kaum erkannt werden.

| Metrik | Zweck |
|---|---|
| **Macro-F1** | Gleichwertige Bewertung aller Klassen |
| **Weighted-F1** | Bewertung entsprechend der Klassenhäufigkeit |
| **Balanced Accuracy** | Durchschnittliche Erkennungsrate aller Klassen |
| **Accuracy** | Anteil aller korrekten Vorhersagen |
| **Metriken pro Klasse** | Erkennung problematischer Klassen |
| **Konfusionsmatrix** | Darstellung typischer Verwechslungen |

In [248]:
min_class_count = y_train.value_counts().min()
n_splits = min(5, min_class_count)

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42,
)

print("Vorbereitete CV-Folds:", n_splits)

Vorbereitete CV-Folds: 4


In [249]:
scoring = {
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
}

main_metric = "macro_f1"

print("Hauptmetrik:", main_metric)
print("Verwendete Metriken:", list(scoring.keys()))

Hauptmetrik: macro_f1
Verwendete Metriken: ['macro_f1', 'weighted_f1', 'balanced_accuracy', 'accuracy']


In [250]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
)


def create_classification_report_df(
    y_true,
    y_pred,
):
    """Erstellt einen DataFrame mit den Metriken pro Klasse."""
    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            output_dict=True,
            zero_division=0,
        )
    ).transpose()

    return report_df


def plot_confusion_matrix(
    y_true,
    y_pred,
    title="Konfusionsmatrix",
    figsize=(14, 14),
):
    """Visualisiert die Verwechslungen zwischen den Klassen."""
    fig, ax = plt.subplots(figsize=figsize)

    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        xticks_rotation=90,
        ax=ax,
    )

    ax.set_title(title)

    plt.tight_layout()
    plt.show()

## 5.3. Geeignete Modellhypothesen für den Politikbereich-Klassifikator

Die Aufgabe ist eine mehrklassige Klassifikation des `politikbereich`. Die Eingabedaten bestehen aus mehreren bereinigten Textspalten, kategorialen Merkmalen und numerischen Informationen. Gleichzeitig ist die Zielvariable deutlich unausgewogen. Deshalb sollten zunächst Modelle gewählt werden, die mit hochdimensionalen, sparsamen Textmerkmalen gut umgehen können und zugleich als robuste Baseline interpretierbar bleiben.

Die Modellhypothesen werden in zwei Gruppen getrennt. **Klassische Modelle** werden zuerst umgesetzt, weil sie schnell, reproduzierbar und gut mit Cross-Validation kombinierbar sind. **NLP- bzw. Deep-Learning-Modelle** werden als spätere Erweiterung betrachtet, da sie deutlich mehr Rechenzeit, zusätzliche Trainingslogik und häufig eine separate Validierungsstrategie benötigen.

#### Klassische Modelle

| Modellkandidat | Begründung für diesen Datensatz | Rolle im Projekt |
|---|---|---|
| **Dummy Classifier** | Sagt nur die häufigste Klasse voraus und nutzt keine Eingabemerkmale. Dadurch entsteht ein einfacher Mindestvergleichswert für alle echten Modelle. | Referenzmodell |
| **Linear SVM / LinearSVC mit TF-IDF** | Textdaten mit TF-IDF erzeugen sehr viele sparse Merkmale. Lineare SVMs gelten in der Literatur als besonders geeignet für hochdimensionale Textklassifikationsprobleme. Es werden Varianten mit und ohne Klassengewichtung sowie unterschiedlicher Regularisierung getestet. | Starker klassischer Benchmark |
| **SGDClassifier mit Hinge Loss** | Approximiert eine lineare SVM mit stochastischem Gradientenverfahren. Diese Variante kann bei großen sparse Textdaten rechnerisch schneller sein als `LinearSVC`. | Effiziente SVM-Alternative |
| **SGDClassifier mit Log Loss** | Verwendet stochastisches Gradientenverfahren für eine logistische Regression. Damit dient das Modell als schnelle, skalierbare Variante der logistischen Regression und kann besonders bei vielen TF-IDF-Merkmalen nützlich sein. | Schnelle logistische Baseline |
| **Logistische Regression mit TF-IDF** | Ebenfalls ein starker linearer Textklassifikator, liefert gut interpretierbare Koeffizienten und unterstützt Klassengewichtung. | Interpretierbare lineare Baseline |
| **Complement Naive Bayes** | Sehr schneller Textklassifikator, der speziell als Verbesserung von Naive Bayes für unausgewogene Textdaten vorgeschlagen wurde. Da numerische und kategoriale Zusatzmerkmale nur eingeschränkt zu Naive Bayes passen, wird dieses Modell als textbasierte Zusatzbaseline betrachtet. | Schnelle einfache Textbaseline |

#### NLP- und Deep-Learning-Modelle

| Modellkandidat | Begründung für diesen Datensatz | Rolle im Projekt |
|---|---|---|
| **fastText** | Effizientes Textklassifikationsmodell auf Basis von Wort- und n-Gramm-Repräsentationen. In der Literatur wird es als sehr schnelle und oft konkurrenzfähige Baseline beschrieben. Da es nicht direkt in die bestehende scikit-learn-Pipeline passt, wird es als separater NLP-Vergleich betrachtet. | Effiziente NLP-Alternative |
| **German BERT / multilingual BERT** | Transformer-Modelle können semantischen Kontext besser erfassen als reine TF-IDF-Modelle. Sie sind jedoch rechenintensiver, benötigen Tokenisierung und Fine-Tuning und sind weniger direkt interpretierbar. | Erweiterte Deep-Learning-Hypothese bei ausreichender Zeit |

Für dieses Projekt ist es sinnvoll, zuerst mit den klassischen TF-IDF-basierten Modellen zu beginnen. Diese Modelle liefern eine transparente Vergleichsbasis und zeigen, wie weit man mit gut kontrollierbaren linearen Verfahren kommt. Im ersten Schritt werden daher ein Dummy Classifier, lineare SVM-Varianten, zwei SGD-basierte lineare Modelle, eine logistische Regression und Complement Naive Bayes verglichen. fastText und BERT werden nicht als erste Baseline, sondern als spätere Erweiterung eingeordnet.

Die SVM-Varianten prüfen den Einfluss von Regularisierung und Klassengewichtung. Der `SGDClassifier(loss="hinge")` wird als rechnerisch effiziente SVM-Alternative aufgenommen. Der `SGDClassifier(loss="log_loss")` ergänzt dies als schnelle logistische Variante: Er optimiert eine logistische Verlustfunktion mit stochastischem Gradientenverfahren und kann deshalb ähnlich wie logistische Regression interpretiert werden, ist aber bei großen sparse Matrizen oft schneller zu trainieren.

Wichtig ist, dass der Modellvergleich nicht primär über Accuracy erfolgt. Wegen der Klassenunverteilung wird der **Macro-F1-Score** als Hauptmetrik verwendet. Zusätzlich werden Weighted-F1, Balanced Accuracy, Metriken pro Klasse, Konfusionsmatrix sowie die Trainingszeit betrachtet.

Literaturhinweise:

- Joachims, T. (1998): *Text Categorization with Support Vector Machines*. Der Beitrag zeigt, warum SVMs für hochdimensionale Textdaten gut geeignet sind. https://www.cs.cornell.edu/people/tj/publications/joachims_98a.pdf
- Fan, R.-E. et al. (2008): *LIBLINEAR: A Library for Large Linear Classification*. Die Arbeit beschreibt effiziente lineare Klassifikationsverfahren für große sparse Daten. https://www.jmlr.org/papers/v9/fan08a.html
- scikit-learn Dokumentation: `SGDClassifier` unterstützt unter anderem `hinge` für lineare SVMs und `log_loss` für logistische Regression mit SGD. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html
- Rennie, J. D. M. et al. (2003): *Tackling the Poor Assumptions of Naive Bayes Text Classifiers*. Die Arbeit schlägt Complement Naive Bayes als robuste Variante für Textklassifikation vor. https://people.csail.mit.edu/jrennie/papers/icml03-nb.pdf
- Joulin, A. et al. (2017): *Bag of Tricks for Efficient Text Classification*. fastText wird als einfache, schnelle und häufig konkurrenzfähige Methode für Textklassifikation vorgestellt. https://aclanthology.org/E17-2068/
- Devlin, J. et al. (2019): *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. BERT bildet die Grundlage für kontextuelle Transformer-Modelle in NLP-Aufgaben. https://aclanthology.org/N19-1423/
- Wahba, Y. et al. (2022): *A Comparison of SVM against Pre-trained Language Models for Text Classification Tasks*. Die Studie zeigt, dass lineare SVMs mit TF-IDF in domänenspezifischen Textklassifikationsaufgaben weiterhin sehr konkurrenzfähig sein können. https://arxiv.org/abs/2211.02563


## 5.4. Definition der Modelle

In diesem Schritt werden die klassischen Modellkandidaten definiert, aber noch nicht trainiert. Die meisten Modelle verwenden dieselbe Vorverarbeitung aus Abschnitt 5.1, damit die Ergebnisse später fair miteinander verglichen werden können.

Der `DummyClassifier` dient als einfache Referenz. Die `LinearSVC`-Modelle bilden starke klassische SVM-Benchmarks. Der `SGDClassifier(loss="hinge")` wird als schnelle SVM-nahe Alternative ergänzt. Zusätzlich wird `SGDClassifier(loss="log_loss")` aufgenommen, weil er eine logistische Regression mit stochastischem Gradientenverfahren trainiert und damit als schnellere Variante der logistischen Regression auf großen sparse TF-IDF-Matrizen dienen kann.

Die reguläre logistische Regression bleibt als interpretierbare lineare Baseline enthalten. `ComplementNB` wird als sehr schnelle textbasierte Zusatzbaseline aufgenommen. Da dieses Modell besonders für sparse, nicht-negative Textmerkmale wie TF-IDF geeignet ist, wird es bewusst nur mit den Textspalten verwendet.


In diesem Schritt werden die ersten Modellkandidaten definiert, aber noch nicht trainiert. Alle Modelle verwenden dieselbe Vorverarbeitung aus Abschnitt 5.1, damit die Ergebnisse später fair miteinander verglichen werden können.

Der `DummyClassifier` dient als einfache Referenz: Er zeigt, welche Leistung ein Modell ohne echte Mustererkennung erreicht. Als SVM-basierte Varianten werden mehrere `LinearSVC`-Modelle definiert. Dabei wird nicht nur die Klassengewichtung variiert, sondern auch die Regularisierung über den Parameter `C`. Ein kleineres `C` bedeutet stärkere Regularisierung und kann bei hochdimensionalen TF-IDF-Merkmalen zu stabilerer Optimierung führen.

Die Variante `Linear SVC C0.1 balanced` wird hier bewusst nicht aufgenommen. Stattdessen werden eine ungewichtete Standardvariante, eine stärker regularisierte ungewichtete Variante, die bereits definierte gewichtete Variante mit `C=0.05` sowie ein SGD-basierter linearer SVM-Ansatz verglichen. `SGDClassifier(loss="hinge")` approximiert eine lineare SVM und kann bei großen sparse Textdaten rechnerisch robuster sein. Die logistische Regression bleibt als interpretierbare lineare Baseline enthalten.

### 5.4.1 Erste Modelle

In [251]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


text_only_preprocessor = ColumnTransformer(
    transformers=[
        (
            f"tfidf_{column}",
            text_transformer,
            [column],
        )
        for column in text_features
    ],
    remainder="drop",
)


models = {
    "Dummy Classifier": Pipeline(
        steps=[
            (
                "classifier",
                DummyClassifier(
                    strategy="most_frequent",
                    random_state=42,
                ),
            ),
        ],
        memory=sklearn_memory,
    ),
    "Linear SVC unweighted": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "classifier",
                LinearSVC(
                    class_weight=None,
                    random_state=42,
                    C=0.1,
                    max_iter=20000,
                    tol=1e-3,
                    dual="auto",
                ),
            ),
        ],
        memory=sklearn_memory,
    ),
    "Linear SVC balanced": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "classifier",
                LinearSVC(
                    class_weight="balanced",
                    random_state=42,
                    C=0.05,
                    max_iter=30000,
                    tol=1e-3,
                    dual="auto",
                ),
            ),
        ],
        memory=sklearn_memory,
    ),
    "SGD hinge balanced": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "classifier",
                SGDClassifier(
                    loss="hinge",
                    penalty="l2",
                    alpha=1e-4,
                    class_weight="balanced",
                    max_iter=1000,
                    tol=1e-3,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ],
        memory=sklearn_memory,
    ),
    "SGD log loss balanced": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "classifier",
                SGDClassifier(
                    loss="log_loss",
                    penalty="l2",
                    alpha=1e-4,
                    class_weight="balanced",
                    max_iter=1000,
                    tol=1e-3,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ],
        memory=sklearn_memory,
    ),
    "Logistic Regression balanced": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    solver="saga",
                    max_iter=3000,
                    tol=1e-2,
                    n_jobs=-1,
                    random_state=42,
                ),
            ),
        ],
        memory=sklearn_memory,
    ),
    "Complement Naive Bayes text only": Pipeline(
        steps=[
            (
                "preprocessor",
                text_only_preprocessor,
            ),
            (
                "classifier",
                ComplementNB(
                    alpha=1.0,
                ),
            ),
        ],
        memory=sklearn_memory,
    ),
}

model_overview = pd.DataFrame(
    [
        {
            "Modell": model_name,
            "Classifier": model.named_steps["classifier"].__class__.__name__,
            "Verwendet_Preprocessing": "preprocessor" in model.named_steps,
        }
        for model_name, model in models.items()
    ]
)

display(model_overview)


,Modell,Classifier,Verwendet_Preprocessing
0,Dummy Classifier,DummyClassifier,False
1,Linear SVC unweighted,LinearSVC,True
2,Linear SVC balanced,LinearSVC,True
3,SGD hinge balanced,SGDClassifier,True
4,SGD log loss balanced,SGDClassifier,True
5,Logistic Regression balanced,LogisticRegression,True
6,Complement Naive Bayes text only,ComplementNB,True


## 5.5. Baseline Cross-Validation mit MLflow

In diesem Abschnitt werden die zuvor definierten Baseline-Modelle mit derselben Cross-Validation und denselben Metriken bewertet. Ziel ist nicht die finale Modelloptimierung, sondern ein fairer erster Vergleich der Modellklassen.

MLflow wird dabei als Experiment-Tracking-Werkzeug verwendet. Es speichert für jeden Modelllauf die wichtigsten Parameter, Metriken und Ergebnisdateien. Dadurch bleiben die Experimente reproduzierbar und später vergleichbar, auch wenn weitere Modelle oder Optuna-Tuning hinzukommen.


### 5.5.1. MLflow vorbereiten

Zuerst wird ein lokaler MLflow-Tracking-Ordner im Projektverzeichnis definiert. Alle Baseline-Ergebnisse werden unter demselben Experimentnamen gespeichert. So lassen sich die Läufe später über die MLflow UI vergleichen.


In [252]:
import tempfile

import mlflow
from sklearn.model_selection import cross_validate


mlflow_tracking_dir = project_root / "mlruns"
mlflow.set_tracking_uri(mlflow_tracking_dir.as_uri())
mlflow.set_experiment("politikbereich_classifier")

print("MLflow Tracking URI:", mlflow.get_tracking_uri())
print("MLflow Experiment: politikbereich_classifier")


MLflow Tracking URI: file:///d:/toydev/schwarz-test/mlruns
MLflow Experiment: politikbereich_classifier


### 5.5.2. Hilfsfunktionen für Logging und Auswertung

Die folgenden Funktionen kapseln die wiederkehrenden Schritte. Für jedes Modell werden die Cross-Validation-Ergebnisse berechnet, Mittelwert und Standardabweichung der Metriken gebildet und anschließend in MLflow gespeichert. Zusätzlich wird die detaillierte Fold-Auswertung als CSV-Artefakt abgelegt.


In [253]:
def create_cv_summary_row(
    model_name,
    cv_results,
):
    """Fasst die Cross-Validation-Ergebnisse eines Modells zusammen."""
    summary = {
        "Modell": model_name,
        "CV_Folds": n_splits,
        "fit_time_total_seconds": round(
            cv_results["fit_time"].sum(),
            2,
        ),
        "fit_time_mean_seconds": round(
            cv_results["fit_time"].mean(),
            2,
        ),
        "fit_time_std_seconds": round(
            cv_results["fit_time"].std(),
            2,
        ),
        "score_time_mean_seconds": round(
            cv_results["score_time"].mean(),
            2,
        ),
    }

    for metric_name in scoring.keys():
        test_scores_for_metric = cv_results[f"test_{metric_name}"]
        summary[f"test_{metric_name}_mean"] = round(
            test_scores_for_metric.mean(),
            4,
        )
        summary[f"test_{metric_name}_std"] = round(
            test_scores_for_metric.std(),
            4,
        )

        # Kurzform für die Hauptsortierung beibehalten.
        summary[f"{metric_name}_mean"] = summary[
            f"test_{metric_name}_mean"
        ]
        summary[f"{metric_name}_std"] = summary[
            f"test_{metric_name}_std"
        ]

        train_metric_key = f"train_{metric_name}"

        if train_metric_key in cv_results:
            train_scores_for_metric = cv_results[train_metric_key]
            summary[f"train_{metric_name}_mean"] = round(
                train_scores_for_metric.mean(),
                4,
            )
            summary[f"train_{metric_name}_std"] = round(
                train_scores_for_metric.std(),
                4,
            )
            summary[f"generalization_gap_{metric_name}"] = round(
                summary[f"train_{metric_name}_mean"]
                - summary[f"test_{metric_name}_mean"],
                4,
            )

    return summary


def create_cv_fold_results_df(
    model_name,
    cv_results,
):
    """Erstellt eine Tabelle mit den Metriken pro CV-Fold."""
    fold_metric_results = {}

    for metric_name in scoring.keys():
        fold_metric_results[f"test_{metric_name}"] = cv_results[
            f"test_{metric_name}"
        ]

        train_metric_key = f"train_{metric_name}"

        if train_metric_key in cv_results:
            fold_metric_results[f"train_{metric_name}"] = cv_results[
                train_metric_key
            ]
            fold_metric_results[
                f"generalization_gap_{metric_name}"
            ] = (
                cv_results[train_metric_key]
                - cv_results[f"test_{metric_name}"]
            )

    fold_results = pd.DataFrame(fold_metric_results)

    fold_results.insert(
        0,
        "score_time_seconds",
        cv_results["score_time"],
    )
    fold_results.insert(
        0,
        "fit_time_seconds",
        cv_results["fit_time"],
    )
    fold_results.insert(
        0,
        "fold",
        range(1, len(fold_results) + 1),
    )
    fold_results.insert(0, "model", model_name)

    return fold_results

def log_baseline_run(
    model_name,
    model,
    summary_row,
    fold_results,
):
    """Speichert Parameter, Metriken und Fold-Ergebnisse in MLflow."""
    classifier = model.named_steps["classifier"]

    with mlflow.start_run(run_name=model_name):
        mlflow.set_tag("stage", "baseline_cross_validation")
        mlflow.set_tag("main_metric", main_metric)

        mlflow.log_params(
            {
                "model_name": model_name,
                "classifier": classifier.__class__.__name__,
                "uses_preprocessor": "preprocessor" in model.named_steps,
                "n_train_rows": len(X_train),
                "n_classes": y_train.nunique(),
                "cv_folds": n_splits,
                "text_features": ", ".join(text_features),
                "categorical_features": ", ".join(categorical_features),
                "numeric_features": ", ".join(numeric_features),
            }
        )

        for parameter_name, parameter_value in classifier.get_params().items():
            if parameter_name in [
                "alpha",
                "class_weight",
                "C",
                "dual",
                "loss",
                "penalty",
                "solver",
                "strategy",
                "max_iter",
                "tol",
                "random_state",
            ]:
                mlflow.log_param(
                    f"classifier__{parameter_name}",
                    str(parameter_value),
                )

        metric_columns = [
            column
            for column in summary_row.index
            if column.endswith("_mean")
            or column.endswith("_std")
            or column.endswith("_seconds")
        ]

        for metric_name in metric_columns:
            metric_value = summary_row[metric_name]

            if pd.notna(metric_value):
                mlflow.log_metric(
                    metric_name,
                    float(metric_value),
                )

        with tempfile.TemporaryDirectory() as temporary_directory:
            artifact_path = (
                Path(temporary_directory)
                / "cv_fold_results.csv"
            )

            fold_results.to_csv(
                artifact_path,
                index=False,
                sep=";",
                encoding="utf-8",
            )

            mlflow.log_artifact(
                str(artifact_path),
                artifact_path="cross_validation",
            )

### 5.5.3. Baseline-Modelle evaluieren

Nun werden alle definierten Baseline-Modelle mit derselben stratifizierten Cross-Validation bewertet. Falls für ein Modell bereits ein gespeichertes Ergebnis existiert, wird dieses wiederverwendet und das Modell wird nicht erneut trainiert. Neue Modelle werden berechnet und anschließend an die bestehenden Ergebnisdateien angehängt.

Zusätzlich werden `fit_time` und `score_time` gespeichert. Dadurch kann nicht nur die Modellgüte, sondern auch der Rechenaufwand der einzelnen Modellvarianten verglichen werden. Der Testdatensatz wird hier weiterhin nicht verwendet. Er bleibt für die finale Bewertung des ausgewählten Modells reserviert.


In [254]:
baseline_results_path = project_root / "data" / "processed" / "baseline_cv_results.csv"

baseline_fold_results_path = (
    project_root / "data" / "processed" / "baseline_cv_fold_results.csv"
)

if baseline_results_path.exists():
    existing_baseline_results = pd.read_csv(
        baseline_results_path,
        sep=";",
        encoding="utf-8",
    )
else:
    existing_baseline_results = pd.DataFrame()

if baseline_fold_results_path.exists():
    existing_fold_results = pd.read_csv(
        baseline_fold_results_path,
        sep=";",
        encoding="utf-8",
    )
else:
    existing_fold_results = pd.DataFrame()

required_summary_columns = [
    "fit_time_total_seconds",
    "fit_time_mean_seconds",
    "fit_time_std_seconds",
    "score_time_mean_seconds",
]

for column in required_summary_columns:
    if column not in existing_baseline_results.columns:
        existing_baseline_results[column] = pd.NA

required_fold_columns = [
    "fit_time_seconds",
    "score_time_seconds",
]

for column in required_fold_columns:
    if column not in existing_fold_results.columns:
        existing_fold_results[column] = pd.NA

existing_model_names = set(
    existing_baseline_results.get(
        "Modell",
        pd.Series(dtype="string"),
    ).dropna()
)

new_baseline_rows = []
new_fold_results = []

for model_name, model in tqdm(
    list(models.items()),
    total=len(models),
    desc="Baseline-Modelle",
):
    if model_name in existing_model_names:
        print("Verwende gespeichertes Ergebnis f\u00fcr:", model_name)
        continue

    print("Starte Cross-Validation f\u00fcr:", model_name)

    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=True,
    )

    summary_row = create_cv_summary_row(
        model_name,
        cv_results,
    )

    model_path = fit_and_save_model(
        model,
        model_name,
        X_train,
        y_train,
        subdirectory="baseline",
    )
    summary_row["model_path"] = str(model_path)
    fold_results = create_cv_fold_results_df(
        model_name,
        cv_results,
    )

    log_baseline_run(
        model_name,
        model,
        pd.Series(summary_row),
        fold_results,
    )

    new_baseline_rows.append(summary_row)
    new_fold_results.append(fold_results)

def concat_non_empty_dataframes(dataframes):
    """Verbindet nur nicht-leere DataFrames und vermeidet pandas-FutureWarnings."""
    non_empty_dataframes = [
        dataframe
        for dataframe in dataframes
        if dataframe is not None and not dataframe.empty
    ]

    if not non_empty_dataframes:
        return pd.DataFrame()

    return pd.concat(
        non_empty_dataframes,
        ignore_index=True,
    )


if new_baseline_rows:
    new_baseline_results = pd.DataFrame(new_baseline_rows)
    baseline_results = concat_non_empty_dataframes(
        [
            existing_baseline_results,
            new_baseline_results,
        ]
    )
else:
    baseline_results = existing_baseline_results.copy()

if new_fold_results:
    baseline_fold_results = concat_non_empty_dataframes(
        [
            existing_fold_results,
            *new_fold_results,
        ]
    )
else:
    baseline_fold_results = existing_fold_results.copy()

baseline_results = (
    baseline_results.drop_duplicates(
        subset=["Modell"],
        keep="last",
    )
    .sort_values(
        [f"{main_metric}_mean", f"{main_metric}_std"],
    ascending=[False, True],
    )
    .reset_index(drop=True)
)

baseline_fold_results = baseline_fold_results.drop_duplicates(
    subset=["model", "fold"],
    keep="last",
).reset_index(drop=True)

display(baseline_results)

Baseline-Modelle: 100%|██████████| 7/7 [00:00<?, ?it/s]

Verwende gespeichertes Ergebnis für: Dummy Classifier
Verwende gespeichertes Ergebnis für: Linear SVC unweighted
Verwende gespeichertes Ergebnis für: Linear SVC balanced
Verwende gespeichertes Ergebnis für: SGD hinge balanced
Verwende gespeichertes Ergebnis für: SGD log loss balanced
Verwende gespeichertes Ergebnis für: Logistic Regression balanced
Verwende gespeichertes Ergebnis für: Complement Naive Bayes text only


,Modell,CV_Folds,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,balanced_accuracy_mean,balanced_accuracy_std,accuracy_mean,accuracy_std,fit_time_total_seconds,fit_time_mean_seconds,fit_time_std_seconds,score_time_mean_seconds
0,Logistic Regression balanced,4,0.8246,0.0060,0.9068,0.0037,0.8533,0.0094,0.9035,0.0041,NaN,NaN,NaN,NaN
1,Linear SVC unweighted,4,0.8225,0.0167,0.9266,0.0014,0.7921,0.0185,0.9281,0.0012,73.86,18.47,1.57,0.65
2,Linear SVC balanced,4,0.8164,0.0071,0.9138,0.0008,0.8432,0.0135,0.9124,0.0008,NaN,NaN,NaN,NaN
3,SGD hinge balanced,4,0.8062,0.0209,0.9220,0.0037,0.8381,0.0209,0.9201,0.0045,10.05,2.51,0.07,0.67
4,SGD log loss balanced,4,0.7793,0.0080,0.8817,0.0031,0.8145,0.0099,0.8779,0.0039,10.08,2.52,0.11,0.70
5,Complement Naive Bayes text only,4,0.6306,0.0041,0.8539,0.0017,0.5931,0.0048,0.8635,0.0017,8.39,2.10,0.07,0.71
6,Dummy Classifier,4,0.0103,0.0000,0.0653,0.0000,0.0312,0.0000,0.1977,0.0000,NaN,NaN,NaN,NaN


### 5.5.4. Ergebnis speichern

Die zusammengefassten Baseline-Ergebnisse werden zusätzlich als CSV-Datei im Projekt gespeichert. Diese Datei kann später im Optuna-Notebook verwendet werden, um zu entscheiden, welche Modelle für das Tuning priorisiert werden.


In [255]:
baseline_results.to_csv(
    baseline_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

baseline_fold_results.to_csv(
    baseline_fold_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

print("Gespeichert:", baseline_results_path)
print("Gespeichert:", baseline_fold_results_path)


Gespeichert: d:\toydev\schwarz-test\data\processed\baseline_cv_results.csv
Gespeichert: d:\toydev\schwarz-test\data\processed\baseline_cv_fold_results.csv


## 5.6. Feature-Ablation und Repräsentationsexperimente

Die folgenden Experimente bauen auf den Beobachtungen aus der explorativen Datenanalyse auf. Besonders `name` und `anschrift` stehen stark mit der Empfänger-ID in Verbindung und können deshalb als indirekte Identifikationsmerkmale wirken. `zweck` beschreibt dagegen den inhaltlichen Förderzweck und ist damit die fachlich naheliegendste Informationsquelle für den Politikbereich. `geber` kann zusätzlich als administrativer Kontext wirken, während `art`, `jahr` und `betrag` strukturierte Zusatzinformationen darstellen.

Es werden zwei Experimenttypen unterschieden. Bei der **Feature-Ablation** werden einzelne Informationsquellen entfernt oder hinzugefügt, um ihren Beitrag zur Modellleistung zu prüfen. Bei den **Repräsentationsexperimenten** bleibt die Information grundsätzlich erhalten, wird aber anders vorverarbeitet, zum Beispiel `geber` als Textmerkmal oder als kategoriales Merkmal. Damit die Ergebnisse interpretierbar bleiben, wird das Modell festgehalten und nur die verwendeten Spalten bzw. deren Repräsentation werden variiert.


### 5.6.1. Experimentdesign

Die Feature-Ablationen prüfen systematisch den Beitrag einzelner Informationsquellen. Die Repräsentationsexperimente untersuchen dagegen, ob `geber` und `art` besser als Textmerkmale oder als kategoriale Merkmale verarbeitet werden sollten. Das ist besonders für `geber` relevant, weil die Spalte zwar nur wenige Ausprägungen enthält, die Namen der Verwaltungseinheiten aber selbst Begriffe wie Kultur, Wirtschaft oder Europa enthalten können.


In [256]:
ablation_configs = [
    {
        "Ablation": "Full model",
        "Experimenttyp": "Feature-Ablation",
        "Beschreibung": "Alle vorgesehenen Merkmalsgruppen",
        "text_features": [
            "name_standardised",
            "geber_standardised",
            "anschrift_standardised",
            "zweck_standardised",
        ],
        "categorical_features": [
            "art_standardised",
            "jahr",
        ],
        "numeric_features": [
            "betrag",
        ],
    },
    {
        "Ablation": "Zweck only",
        "Experimenttyp": "Feature-Ablation",
        "Beschreibung": "Nur inhaltlicher Förderzweck",
        "text_features": [
            "zweck_standardised",
        ],
        "categorical_features": [],
        "numeric_features": [],
    },
    {
        "Ablation": "Zweck + geber text",
        "Experimenttyp": "Repräsentationsexperiment",
        "Beschreibung": "Förderzweck plus Geber als Text",
        "text_features": [
            "zweck_standardised",
            "geber_standardised",
        ],
        "categorical_features": [],
        "numeric_features": [],
    },
    {
        "Ablation": "Zweck + geber category",
        "Experimenttyp": "Repräsentationsexperiment",
        "Beschreibung": "Förderzweck plus Geber als Kategorie",
        "text_features": [
            "zweck_standardised",
        ],
        "categorical_features": [
            "geber_standardised",
        ],
        "numeric_features": [],
    },
    {
        "Ablation": "Ohne Empfängeridentität",
        "Experimenttyp": "Feature-Ablation",
        "Beschreibung": "Ohne Name und Anschrift",
        "text_features": [
            "geber_standardised",
            "zweck_standardised",
        ],
        "categorical_features": [
            "art_standardised",
            "jahr",
        ],
        "numeric_features": [
            "betrag",
        ],
    },
    {
        "Ablation": "Ohne Anschrift",
        "Experimenttyp": "Feature-Ablation",
        "Beschreibung": "Alle Merkmale außer Anschrift",
        "text_features": [
            "name_standardised",
            "geber_standardised",
            "zweck_standardised",
        ],
        "categorical_features": [
            "art_standardised",
            "jahr",
        ],
        "numeric_features": [
            "betrag",
        ],
    },
    {
        "Ablation": "Ohne Name",
        "Experimenttyp": "Feature-Ablation",
        "Beschreibung": "Alle Merkmale außer Name",
        "text_features": [
            "geber_standardised",
            "anschrift_standardised",
            "zweck_standardised",
        ],
        "categorical_features": [
            "art_standardised",
            "jahr",
        ],
        "numeric_features": [
            "betrag",
        ],
    },
    {
        "Ablation": "Text only",
        "Experimenttyp": "Feature-Ablation",
        "Beschreibung": "Nur Textspalten ohne strukturierte Zusatzmerkmale",
        "text_features": [
            "name_standardised",
            "geber_standardised",
            "anschrift_standardised",
            "zweck_standardised",
        ],
        "categorical_features": [],
        "numeric_features": [],
    },
    {
        "Ablation": "Text + categorical",
        "Experimenttyp": "Feature-Ablation",
        "Beschreibung": "Textspalten plus art und jahr",
        "text_features": [
            "name_standardised",
            "geber_standardised",
            "anschrift_standardised",
            "zweck_standardised",
        ],
        "categorical_features": [
            "art_standardised",
            "jahr",
        ],
        "numeric_features": [],
    },
    {
        "Ablation": "Full with geber category",
        "Experimenttyp": "Repräsentationsexperiment",
        "Beschreibung": "Geber als Kategorie statt als Text",
        "text_features": [
            "name_standardised",
            "anschrift_standardised",
            "zweck_standardised",
        ],
        "categorical_features": [
            "geber_standardised",
            "art_standardised",
            "jahr",
        ],
        "numeric_features": [
            "betrag",
        ],
    },
    {
        "Ablation": "Full with geber and art as text",
        "Experimenttyp": "Repräsentationsexperiment",
        "Beschreibung": "Geber und art beide als Text",
        "text_features": [
            "name_standardised",
            "geber_standardised",
            "art_standardised",
            "anschrift_standardised",
            "zweck_standardised",
        ],
        "categorical_features": [
            "jahr",
        ],
        "numeric_features": [
            "betrag",
        ],
    },
]

ablation_overview = pd.DataFrame(
    [
        {
            "Ablation": config["Ablation"],
            "Experimenttyp": config["Experimenttyp"],
            "Beschreibung": config["Beschreibung"],
            "Textspalten": ", ".join(config["text_features"]),
            "Kategoriale Spalten": ", ".join(config["categorical_features"]),
            "Numerische Spalten": ", ".join(config["numeric_features"]),
        }
        for config in ablation_configs
    ]
)

display(
    ablation_overview.set_index("Ablation")
)


,Experimenttyp,Beschreibung,Textspalten,Kategoriale Spalten,Numerische Spalten
Ablation,,,,,
Full model,Feature-Ablation,Alle vorgesehenen Merkmalsgruppen,"name_standardised, geber_standardised, anschri...","art_standardised, jahr",betrag
Zweck only,Feature-Ablation,Nur inhaltlicher Förderzweck,zweck_standardised,,
Zweck + geber text,Repräsentationsexperiment,Förderzweck plus Geber als Text,"zweck_standardised, geber_standardised",,
Zweck + geber category,Repräsentationsexperiment,Förderzweck plus Geber als Kategorie,zweck_standardised,geber_standardised,
Ohne Empfängeridentität,Feature-Ablation,Ohne Name und Anschrift,"geber_standardised, zweck_standardised","art_standardised, jahr",betrag
Ohne Anschrift,Feature-Ablation,Alle Merkmale außer Anschrift,"name_standardised, geber_standardised, zweck_s...","art_standardised, jahr",betrag
Ohne Name,Feature-Ablation,Alle Merkmale außer Name,"geber_standardised, anschrift_standardised, zw...","art_standardised, jahr",betrag
Text only,Feature-Ablation,Nur Textspalten ohne strukturierte Zusatzmerkmale,"name_standardised, geber_standardised, anschri...",,
Text + categorical,Feature-Ablation,Textspalten plus art und jahr,"name_standardised, geber_standardised, anschri...","art_standardised, jahr",


### 5.6.2. Festes Modell für die Experimente auswählen

Für die Feature-Ablation und die Repräsentationsexperimente wird ein Modell festgehalten. Da diese Experimente viele Varianten trainieren, wird nicht automatisch das insgesamt beste Modell gewählt, sondern das beste Modell innerhalb der schnell trainierbaren Baseline-Modelle. Dadurch bleibt die Laufzeit kontrollierbar, während die Auswahl weiterhin datenbasiert über die Hauptmetrik `macro_f1` erfolgt.

Als schnelle Modelle gelten hier Modelle mit einer durchschnittlichen Trainingszeit pro Cross-Validation-Fold von höchstens `max_fast_fit_time_seconds`. Falls für einzelne Modelle noch keine Laufzeit gespeichert wurde, werden bevorzugt die explizit schnellen linearen SGD-Modelle und `ComplementNB` berücksichtigt.


In [257]:
from sklearn.base import clone


max_fast_fit_time_seconds = 5.0
preferred_fast_models = [
    "SGD log loss balanced",
    "SGD hinge balanced",
    "Complement Naive Bayes text only",
]

if "baseline_results" not in globals() or baseline_results.empty:
    raise ValueError(
        "Für die Modellauswahl müssen zuerst die Baseline-Ergebnisse berechnet werden."
    )

baseline_model_candidates = baseline_results[
    baseline_results["Modell"].isin(models.keys())
    & ~baseline_results["Modell"].eq("Dummy Classifier")
].copy()

baseline_model_candidates["fit_time_mean_seconds"] = pd.to_numeric(
    baseline_model_candidates.get(
        "fit_time_mean_seconds",
        pd.Series(
            index=baseline_model_candidates.index,
            dtype="float64",
        ),
    ),
    errors="coerce",
)

measured_fast_candidates = baseline_model_candidates[
    baseline_model_candidates["fit_time_mean_seconds"].le(
        max_fast_fit_time_seconds
    )
].copy()

if measured_fast_candidates.empty:
    fast_model_candidates = baseline_model_candidates[
        baseline_model_candidates["Modell"].isin(
            preferred_fast_models
        )
    ].copy()
else:
    fast_model_candidates = measured_fast_candidates.copy()

if fast_model_candidates.empty:
    raise ValueError(
        "Es wurde kein schnelles Baseline-Modell für die Experimente gefunden."
    )

fast_model_candidates = fast_model_candidates.sort_values(
    [
        f"{main_metric}_mean",
        "fit_time_mean_seconds",
    ],
    ascending=[
        False,
        True,
    ],
    na_position="last",
).reset_index(drop=True)

ablation_model_name = fast_model_candidates.loc[
    0,
    "Modell",
]

base_ablation_classifier = clone(
    models[ablation_model_name].named_steps["classifier"]
)

print("Maximale durchschnittliche Trainingszeit:", max_fast_fit_time_seconds)
print("Schnelle Modellkandidaten:")
display(
    fast_model_candidates[
        [
            "Modell",
            f"{main_metric}_mean",
            "fit_time_mean_seconds",
        ]
    ]
)
print("Modell für Feature-Ablation und Repräsentationsexperimente:", ablation_model_name)
print("Classifier:", base_ablation_classifier.__class__.__name__)


Maximale durchschnittliche Trainingszeit: 5.0
Schnelle Modellkandidaten:


,Modell,macro_f1_mean,fit_time_mean_seconds
0,SGD hinge balanced,0.8062,2.51
1,SGD log loss balanced,0.7793,2.52
2,Complement Naive Bayes text only,0.6306,2.10


Modell für Feature-Ablation und Repräsentationsexperimente: SGD hinge balanced
Classifier: SGDClassifier


### 5.6.3. Experiment-Pipeline erzeugen

Für jede Variante wird ein eigener `ColumnTransformer` erzeugt. Dadurch können dieselben Spalten je nach Testbedingung unterschiedlich verarbeitet werden, zum Beispiel `geber` als Text oder als kategoriales Merkmal.


In [258]:
def build_ablation_preprocessor(
    selected_text_features,
    selected_categorical_features,
    selected_numeric_features,
):
    """Erstellt einen Preprocessor für eine Ablationsvariante."""
    transformers = []

    transformers.extend(
        [
            (
                f"tfidf_{column}",
                text_transformer,
                [column],
            )
            for column in selected_text_features
        ]
    )

    if selected_categorical_features:
        transformers.append(
            (
                "categorical",
                categorical_transformer,
                selected_categorical_features,
            )
        )

    if selected_numeric_features:
        transformers.append(
            (
                "numeric",
                numeric_transformer,
                selected_numeric_features,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


def build_ablation_model(config):
    """Kombiniert Ablations-Preprocessor und festes Modell."""
    ablation_preprocessor = build_ablation_preprocessor(
        config["text_features"],
        config["categorical_features"],
        config["numeric_features"],
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                ablation_preprocessor,
            ),
            (
                "classifier",
                clone(base_ablation_classifier),
            ),
        ],
        memory=sklearn_memory,
    )


### 5.6.4. Cross-Validation und MLflow ausführen

Die Ergebnisse werden wie die Baseline-Ergebnisse zwischengespeichert. Bereits vorhandene Varianten werden nicht erneut trainiert. Neue Varianten werden berechnet, an die bestehenden Ergebnisdateien angehängt und zusätzlich in MLflow dokumentiert.


In [259]:
ablation_results_path = (
    project_root
    / "data"
    / "processed"
    / "ablation_cv_results.csv"
)

ablation_fold_results_path = (
    project_root
    / "data"
    / "processed"
    / "ablation_cv_fold_results.csv"
)

if ablation_results_path.exists():
    existing_ablation_results = pd.read_csv(
        ablation_results_path,
        sep=";",
        encoding="utf-8",
    )
else:
    existing_ablation_results = pd.DataFrame()

if ablation_fold_results_path.exists():
    existing_ablation_fold_results = pd.read_csv(
        ablation_fold_results_path,
        sep=";",
        encoding="utf-8",
    )
else:
    existing_ablation_fold_results = pd.DataFrame()

if (
    not existing_ablation_results.empty
    and "Basis_Modell" in existing_ablation_results.columns
):
    valid_existing_ablation_results = existing_ablation_results[
        existing_ablation_results["Basis_Modell"].eq(
            ablation_model_name
        )
    ].copy()
else:
    valid_existing_ablation_results = pd.DataFrame()

existing_ablation_names = set(
    valid_existing_ablation_results.get(
        "Ablation",
        pd.Series(dtype="string"),
    )
    .dropna()
)

experiment_type_by_name = {
    config["Ablation"]: config["Experimenttyp"]
    for config in ablation_configs
}

if (
    not valid_existing_ablation_results.empty
    and "Ablation" in valid_existing_ablation_results.columns
):
    valid_existing_ablation_results["Experimenttyp"] = (
        valid_existing_ablation_results.get(
            "Experimenttyp",
            pd.Series(
                index=valid_existing_ablation_results.index,
                dtype="string",
            ),
        )
        .fillna(
            valid_existing_ablation_results["Ablation"].map(
                experiment_type_by_name
            )
        )
    )

if (
    not existing_ablation_fold_results.empty
    and "Ablation" in existing_ablation_fold_results.columns
):
    existing_ablation_fold_results["Experimenttyp"] = (
        existing_ablation_fold_results.get(
            "Experimenttyp",
            pd.Series(
                index=existing_ablation_fold_results.index,
                dtype="string",
            ),
        )
        .fillna(
            existing_ablation_fold_results["Ablation"].map(
                experiment_type_by_name
            )
        )
    )

new_ablation_rows = []
new_ablation_fold_results = []

for config in tqdm(
    ablation_configs,
    total=len(ablation_configs),
    desc="Feature-Ablation und Repräsentation",
):
    ablation_name = config["Ablation"]

    if ablation_name in existing_ablation_names:
        print("Verwende gespeichertes Ergebnis für:", ablation_name)
        continue

    print("Starte Ablation für:", ablation_name)

    selected_features = (
        config["text_features"]
        + config["categorical_features"]
        + config["numeric_features"]
    )

    X_ablation = X_train[selected_features].copy()
    ablation_model = build_ablation_model(config)

    cv_results = cross_validate(
        ablation_model,
        X_ablation,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=True,
    )

    summary_row = create_cv_summary_row(
        ablation_name,
        cv_results,
    )

    model_path = fit_and_save_model(
        ablation_model,
        f"{ablation_model_name}__{ablation_name}",
        X_ablation,
        y_train,
        subdirectory="ablation",
    )
    summary_row["model_path"] = str(model_path)
    summary_row["Ablation"] = summary_row.pop("Modell")
    summary_row["Basis_Modell"] = ablation_model_name
    summary_row["Textspalten"] = ", ".join(config["text_features"])
    summary_row["Kategoriale_Spalten"] = ", ".join(
        config["categorical_features"]
    )
    summary_row["Numerische_Spalten"] = ", ".join(
        config["numeric_features"]
    )
    summary_row["Experimenttyp"] = config["Experimenttyp"]
    summary_row["Beschreibung"] = config["Beschreibung"]

    fold_results = create_cv_fold_results_df(
        ablation_name,
        cv_results,
    ).rename(columns={"model": "Ablation"})
    fold_results.insert(1, "Basis_Modell", ablation_model_name)
    fold_results.insert(2, "Experimenttyp", config["Experimenttyp"])

    with mlflow.start_run(run_name=f"Ablation - {ablation_name}"):
        mlflow.set_tag("stage", "feature_ablation_and_representation")
        mlflow.set_tag("experiment_type", config["Experimenttyp"])
        mlflow.set_tag("main_metric", main_metric)
        mlflow.log_params(
            {
                "experiment_name": ablation_name,
                "experiment_type": config["Experimenttyp"],
                "base_model": ablation_model_name,
                "text_features": summary_row["Textspalten"],
                "categorical_features": summary_row[
                    "Kategoriale_Spalten"
                ],
                "numeric_features": summary_row["Numerische_Spalten"],
                "cv_folds": n_splits,
            }
        )

        for metric_name, metric_value in pd.Series(summary_row).items():
            if (
                metric_name.endswith("_mean")
                or metric_name.endswith("_std")
                or metric_name.endswith("_seconds")
            ) and pd.notna(metric_value):
                mlflow.log_metric(metric_name, float(metric_value))

        with tempfile.TemporaryDirectory() as temporary_directory:
            artifact_path = Path(temporary_directory) / "ablation_fold_results.csv"
            fold_results.to_csv(
                artifact_path,
                index=False,
                sep=";",
                encoding="utf-8",
            )
            mlflow.log_artifact(
                str(artifact_path),
                artifact_path="cross_validation",
            )

    new_ablation_rows.append(summary_row)
    new_ablation_fold_results.append(fold_results)

if new_ablation_rows:
    ablation_results = pd.concat(
        [
            valid_existing_ablation_results,
            pd.DataFrame(new_ablation_rows),
        ],
        ignore_index=True,
    )
else:
    ablation_results = existing_ablation_results.copy()

if new_ablation_fold_results:
    ablation_fold_results = pd.concat(
        [
            existing_ablation_fold_results,
            *new_ablation_fold_results,
        ],
        ignore_index=True,
    )
else:
    ablation_fold_results = existing_ablation_fold_results.copy()

ablation_results = (
    ablation_results
    .drop_duplicates(
        subset=["Ablation"],
        keep="last",
    )
    .sort_values(
        f"{main_metric}_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

ablation_fold_results = (
    ablation_fold_results
    .drop_duplicates(
        subset=["Ablation", "fold"],
        keep="last",
    )
    .reset_index(drop=True)
)

ablation_metric_columns = [
    column
    for column in ablation_results.columns
    if column in [
        "Experimenttyp",
        "Basis_Modell",
    ]
    or column.endswith("_mean")
    or column.endswith("_std")
    or column.startswith("generalization_gap_")
]

display(
    ablation_results
    .set_index("Ablation")
    [ablation_metric_columns]
)


Feature-Ablation und Repräsentation: 100%|██████████| 11/11 [00:00<00:00, 20633.87it/s]

Verwende gespeichertes Ergebnis für: Full model
Verwende gespeichertes Ergebnis für: Zweck only
Verwende gespeichertes Ergebnis für: Zweck + geber text
Verwende gespeichertes Ergebnis für: Zweck + geber category
Verwende gespeichertes Ergebnis für: Ohne Empfängeridentität
Verwende gespeichertes Ergebnis für: Ohne Anschrift
Verwende gespeichertes Ergebnis für: Ohne Name
Verwende gespeichertes Ergebnis für: Text only
Verwende gespeichertes Ergebnis für: Text + categorical
Verwende gespeichertes Ergebnis für: Full with geber category
Verwende gespeichertes Ergebnis für: Full with geber and art as text


,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,balanced_accuracy_mean,balanced_accuracy_std,accuracy_mean,accuracy_std,Basis_Modell,Experimenttyp,...,test_balanced_accuracy_mean,test_balanced_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,generalization_gap_balanced_accuracy,test_accuracy_mean,test_accuracy_std,train_accuracy_mean,train_accuracy_std,generalization_gap_accuracy
Ablation,,,,,,,,,,,,,,,,,,,,,
Text + categorical,0.8170,0.0132,0.9229,0.0009,0.8548,0.0095,0.9209,0.0016,SGD hinge balanced,Feature-Ablation,...,0.8548,0.0095,0.9732,0.0031,0.1184,0.9209,0.0016,0.9669,0.0024,0.0460
Full with geber category,0.8117,0.0147,0.9231,0.0023,0.8468,0.0111,0.9216,0.0025,SGD hinge balanced,Repräsentationsexperiment,...,0.8468,0.0111,0.9673,0.0081,0.1205,0.9216,0.0025,0.9669,0.0024,0.0453
Full model,0.8062,0.0209,0.9220,0.0037,0.8381,0.0209,0.9201,0.0045,SGD hinge balanced,Feature-Ablation,...,0.8381,0.0209,0.9605,0.0166,0.1224,0.9201,0.0045,0.9657,0.0045,0.0456
Text only,0.8020,0.0080,0.9256,0.0016,0.8574,0.0137,0.9240,0.0019,SGD hinge balanced,Feature-Ablation,...,0.8574,0.0137,0.9762,0.0013,0.1188,0.9240,0.0019,0.9699,0.0010,0.0459
Ohne Name,0.7965,0.0177,0.9169,0.0040,0.8356,0.0141,0.9151,0.0042,SGD hinge balanced,Feature-Ablation,...,0.8356,0.0141,0.9628,0.0084,0.1272,0.9151,0.0042,0.9632,0.0013,0.0481
Full with geber and art as text,0.7954,0.0210,0.9228,0.0043,0.8482,0.0121,0.9206,0.0059,SGD hinge balanced,Repräsentationsexperiment,...,0.8482,0.0121,0.9613,0.0202,0.1131,0.9206,0.0059,0.9654,0.0058,0.0448
Ohne Anschrift,0.7918,0.0191,0.9210,0.0023,0.8296,0.0196,0.9189,0.0033,SGD hinge balanced,Feature-Ablation,...,0.8296,0.0196,0.9581,0.0124,0.1285,0.9189,0.0033,0.9630,0.0034,0.0441
Ohne Empfängeridentität,0.7664,0.0192,0.8970,0.0038,0.8149,0.0170,0.8939,0.0043,SGD hinge balanced,Feature-Ablation,...,0.8149,0.0170,0.9445,0.0075,0.1296,0.8939,0.0043,0.9474,0.0050,0.0535
Zweck + geber category,0.7525,0.0073,0.9054,0.0014,0.8188,0.0163,0.9031,0.0020,SGD hinge balanced,Repräsentationsexperiment,...,0.8188,0.0163,0.9539,0.0049,0.1351,0.9031,0.0020,0.9574,0.0014,0.0543


In [260]:
# Bestes Szenario anhand der Hauptmetrik auswählen.
best_scenario = (
    ablation_results
    .sort_values(
        f"{main_metric}_mean",
        ascending=False,
    )
    .iloc[0]
)

best_scenario_name = best_scenario["Ablation"]

best_scenario_config = next(
    config
    for config in ablation_configs
    if config["Ablation"] == best_scenario_name
)

best_scenario_columns = (
    best_scenario_config["text_features"]
    + best_scenario_config["categorical_features"]
    + best_scenario_config["numeric_features"]
)

best_scenario_columns_df = pd.DataFrame(
    {
        "Trainingsspalte": best_scenario_columns
    }
)

all_model_columns = (
    text_features
    + categorical_features
    + numeric_features
)

missing_columns = [
    column
    for column in all_model_columns
    if column not in best_scenario_columns
]

missing_columns_df = pd.DataFrame(
    {
        "Nicht verwendete Trainingsspalte": missing_columns
    }
)

print("Bestes Szenario:", best_scenario_name)
print(
    "Macro-F1:",
    round(best_scenario[f"{main_metric}_mean"], 4),
)

display(best_scenario_columns_df)


print("Bestes Szenario:", best_scenario_name)

print(
    "Anzahl verwendeter Spalten:",
    len(best_scenario_columns),
)

print(
    "Anzahl nicht verwendeter Spalten:",
    len(missing_columns),
)

display(missing_columns_df)

Bestes Szenario: Text + categorical
Macro-F1: 0.817


,Trainingsspalte
0,name_standardised
1,geber_standardised
2,anschrift_standardised
3,zweck_standardised
4,art_standardised
5,jahr


Bestes Szenario: Text + categorical
Anzahl verwendeter Spalten: 6
Anzahl nicht verwendeter Spalten: 1


,Nicht verwendete Trainingsspalte
0,betrag


### 5.6.5. Kontrollvergleich mit Logistischer Regression

Zur Prüfung der Ablationsergebnisse wird zusätzlich ein Kontrollvergleich durchgeführt. Dabei bleibt das Modell konstant: Es wird immer `Logistic Regression balanced` verwendet. Verändert werden nur die verwendeten Eingabespalten. Verglichen wird das vollständige Ausgangsszenario mit dem besten Szenario aus der Feature-Ablation bzw. den Repräsentationsexperimenten.

Damit lässt sich prüfen, ob die beobachtete Verbesserung tatsächlich auf die veränderte Datengrundlage zurückzuführen ist und nicht nur auf den zuvor verwendeten schnelleren Modelltyp.


In [261]:
logreg_control_model_name = "Logistic Regression balanced"

if logreg_control_model_name not in models:
    raise KeyError(
        f"Das Kontrollmodell '{logreg_control_model_name}' wurde nicht in models definiert."
    )

logreg_control_classifier = clone(
    models[logreg_control_model_name].named_steps["classifier"]
)

best_scenario_row = (
    ablation_results
    .sort_values(
        f"{main_metric}_mean",
        ascending=False,
    )
    .iloc[0]
)

best_scenario_name = best_scenario_row["Ablation"]

full_scenario_config = next(
    config
    for config in ablation_configs
    if config["Ablation"] == "Full model"
)

best_scenario_config = next(
    config
    for config in ablation_configs
    if config["Ablation"] == best_scenario_name
)

logreg_control_configs = [
    (
        "Full model",
        full_scenario_config,
    ),
]

if best_scenario_name != "Full model":
    logreg_control_configs.append(
        (
            "Best scenario",
            best_scenario_config,
        )
    )
else:
    print(
        "Das beste Szenario entspricht bereits dem vollständigen Modell."
    )

logreg_control_rows = []

for scenario_label, scenario_config in tqdm(
    logreg_control_configs,
    total=len(logreg_control_configs),
    desc="LogReg-Kontrollvergleich",
):
    scenario_features = (
        scenario_config["text_features"]
        + scenario_config["categorical_features"]
        + scenario_config["numeric_features"]
    )

    scenario_preprocessor = build_ablation_preprocessor(
        scenario_config["text_features"],
        scenario_config["categorical_features"],
        scenario_config["numeric_features"],
    )

    scenario_model = Pipeline(
        steps=[
            (
                "preprocessor",
                scenario_preprocessor,
            ),
            (
                "classifier",
                clone(logreg_control_classifier),
            ),
        ],
        memory=sklearn_memory,
    )

    scenario_cv_results = cross_validate(
        scenario_model,
        X_train[scenario_features].copy(),
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=True,
    )

    scenario_summary = create_cv_summary_row(
        scenario_label,
        scenario_cv_results,
    )

    model_path = fit_and_save_model(
        scenario_model,
        f"{logreg_control_model_name}__{scenario_label}",
        X_train[scenario_features].copy(),
        y_train,
        subdirectory="control_comparison",
    )
    scenario_summary["model_path"] = str(model_path)

    scenario_summary["Szenario"] = scenario_label
    scenario_summary["Ablation"] = scenario_config["Ablation"]
    scenario_summary["Experimenttyp"] = scenario_config["Experimenttyp"]
    scenario_summary["Modell"] = logreg_control_model_name
    scenario_summary["Anzahl_Trainingsspalten"] = len(
        scenario_features
    )
    scenario_summary["Trainingsspalten"] = ", ".join(
        scenario_features
    )

    logreg_control_rows.append(scenario_summary)

logreg_control_results = pd.DataFrame(logreg_control_rows)

if "Full model" in logreg_control_results["Szenario"].values:
    full_metric_value = logreg_control_results.loc[
        logreg_control_results["Szenario"].eq("Full model"),
        f"{main_metric}_mean",
    ].iloc[0]

    logreg_control_results[
        f"Differenz_zu_Full_model_{main_metric}"
    ] = (
        logreg_control_results[f"{main_metric}_mean"]
        - full_metric_value
    ).round(4)

logreg_control_metric_columns = [
    column
    for column in logreg_control_results.columns
    if column in [
        "Modell",
        "Ablation",
        "Experimenttyp",
        "Anzahl_Trainingsspalten",
        f"Differenz_zu_Full_model_{main_metric}",
    ]
    or column.endswith("_mean")
    or column.endswith("_std")
    or column.startswith("generalization_gap_")
]

display(
    logreg_control_results
    .set_index("Szenario")
    [logreg_control_metric_columns]
)

best_scenario_train_columns = pd.DataFrame(
    {
        "Trainingsspalte": (
            best_scenario_config["text_features"]
            + best_scenario_config["categorical_features"]
            + best_scenario_config["numeric_features"]
        )
    }
)

print("Bestes Szenario:", best_scenario_name)
display(best_scenario_train_columns)


LogReg-Kontrollvergleich:   0%|          | 0/2 [00:00<?, ?it/s]

LogReg-Kontrollvergleich:  50%|█████     | 1/2 [04:05<04:05, 245.73s/it]


KeyboardInterrupt: 

### 5.6.6. Ergebnisse speichern

Die Ergebnisse werden gespeichert, damit spätere Notebook-Läufe bereits berechnete Varianten wiederverwenden können. Die besten Merkmals- und Repräsentationsentscheidungen dienen anschließend als Grundlage für die Hyperparameteroptimierung mit Optuna.


In [ ]:
ablation_results.to_csv(
    ablation_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

ablation_fold_results.to_csv(
    ablation_fold_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

print("Gespeichert:", ablation_results_path)
print("Gespeichert:", ablation_fold_results_path)


Gespeichert: d:\toydev\schwarz-test\data\processed\ablation_cv_results.csv
Gespeichert: d:\toydev\schwarz-test\data\processed\ablation_cv_fold_results.csv
